# 第二十三课:Mini-GPT 反向传播手撕——数值梯度当裁判

讲义:[23-backward.md](23-backward.md)。本课给第 20 课的 Mini-GPT 手撕整机反向传播,数值梯度(中心差分)逐模块 + 全参数 33,792 坐标验收。

In [ ]:
import math
import time
import numpy as np

# ===== 第 20 课 MiniGPT 原码(逐字复制,作为被验证基准) =====
def softmax(x, axis=-1):
    m = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - m)
    return e / e.sum(axis=axis, keepdims=True)

class MiniGPT:
    def __init__(self, vocab=16, d=32, n_heads=4, n_layers=2, seed=0):
        rng = np.random.default_rng(seed)
        s = 0.1
        self.E = rng.normal(size=(vocab, d)) * 0.5
        self.dh = d // n_heads
        self.H, self.D, self.NL = n_heads, d, n_layers
        self.layers = [{
            'Wq': rng.normal(size=(d, d))*s, 'Wk': rng.normal(size=(d, d))*s,
            'Wv': rng.normal(size=(d, d))*s, 'Wo': rng.normal(size=(d, d))*s,
            'W1': rng.normal(size=(d, 4*d))*s*0.8, 'W2': rng.normal(size=(4*d, d))*s*0.8,
            'W3': rng.normal(size=(d, 4*d))*s*0.8,
        } for _ in range(n_layers)]
        self.head = rng.normal(size=(d, vocab)) * s

    def rms(self, x, eps=1e-5):
        return x / np.sqrt((x**2).mean(-1, keepdims=True) + eps)

    def rope(self, x, pos):
        half = x.shape[-1] // 2
        ang = pos[:, None] / (10000 ** (np.arange(half)*2/x.shape[-1]))
        c, s = np.cos(ang), np.sin(ang)
        x1, x2 = x[..., :half], x[..., half:]
        return np.concatenate([x1*c - x2*s, x1*s + x2*c], -1)

    def _attn(self, x, pos, W, cache):
        n = x.shape[0]
        q, k, v = self.rope(x@W['Wq'], pos), self.rope(x@W['Wk'], pos), x@W['Wv']
        if cache is not None:
            k, v = np.concatenate([cache[0], k]), np.concatenate([cache[1], v])
        T = k.shape[0]
        mask = np.triu(np.ones((n, T), bool), T - n + 1)
        outs = []
        for h in range(self.H):
            sl = slice(h*self.dh, (h+1)*self.dh)
            s = q[:, sl] @ k[:, sl].T / math.sqrt(self.dh)
            s = np.where(mask, -np.inf, s)
            outs.append(softmax(s) @ v[:, sl])
        return np.concatenate(outs, -1) @ W['Wo'], (k, v)

    def _ffn(self, x, W):
        h = x @ W['W1']
        return (h / (1 + np.exp(-h)) * (x @ W['W3'])) @ W['W2']

    def forward(self, ids, cache=None):
        pos0 = cache[0][0].shape[0] if cache else 0
        x = self.E[list(ids)]
        new_cache = []
        for i, W in enumerate(self.layers):
            a, kv = self._attn(x, np.arange(pos0, pos0+len(ids)), W,
                               cache[i] if cache else None)
            x = x + a
            x = x + self._ffn(self.rms(x), W)
            new_cache.append(kv)
        logits = self.rms(x) @ self.head
        assert np.isfinite(logits).all(), 'NaN! 查掩码/精度/ε'
        return logits, new_cache

In [ ]:
# ===== 本课新增:四个局部梯度 + 稳定 CE =====
def rms_backward(dy, x, eps=1e-5):
    """y = x/rms(x):dx = (dy − y·mean(dy⊙y))/r。ε 通过 r 进公式。"""
    r = np.sqrt((x**2).mean(-1, keepdims=True) + eps)
    y = x / r
    return (dy - y * (dy * y).mean(-1, keepdims=True)) / r

def rope_backward(dy, pos, dim):
    """前向是正交旋转 R(m),反向 = R(m)ᵀ:角度取负再转一次。"""
    half = dim // 2
    ang = pos[:, None] / (10000 ** (np.arange(half)*2/dim))
    c, s = np.cos(ang), np.sin(ang)
    dy1, dy2 = dy[..., :half], dy[..., half:]
    return np.concatenate([dy1*c + dy2*s, -dy1*s + dy2*c], -1)

def silu_grad(u):
    sig = 1 / (1 + np.exp(-u))
    return sig * (1 + u * (1 - sig))

def ce_loss(logits, targets):
    z = logits - logits.max(-1, keepdims=True)
    logp = z - np.log(np.exp(z).sum(-1, keepdims=True))
    return -logp[np.arange(len(targets)), targets].mean()

In [ ]:
def loss_and_grads(gpt, ids, targets):
    """带 tape 前向 + 整机反向。返回 (loss, grads)。
    grads = {'E', 'head', 'layers': [ {Wq..W3} ]},键与形状和 gpt 权重一一对应。"""
    T = len(ids)
    pos = np.arange(T)
    tape = []
    x = gpt.E[list(ids)]
    for W in gpt.layers:
        q, k, v = gpt.rope(x@W['Wq'], pos), gpt.rope(x@W['Wk'], pos), x@W['Wv']
        mask = np.triu(np.ones((T, T), bool), 1)          # 第 18 课的差一教训
        ps, outs = [], []
        for h in range(gpt.H):
            sl = slice(h*gpt.dh, (h+1)*gpt.dh)
            s = q[:, sl] @ k[:, sl].T / math.sqrt(gpt.dh)
            s = np.where(mask, -np.inf, s)
            p = softmax(s)                                 # tape:逐头概率
            ps.append(p); outs.append(p @ v[:, sl])
        concat_o = np.concatenate(outs, -1)
        x_mid = x + concat_o @ W['Wo']                     # tape:残差汇合点
        x_rms = gpt.rms(x_mid)
        u, zv = x_rms @ W['W1'], x_rms @ W['W3']
        gate = u / (1 + np.exp(-u)) * zv                   # SwiGLU
        x_out = x_mid + gate @ W['W2']
        tape.append(dict(x_in=x, x_mid=x_mid, x_rms=x_rms, u=u, zv=zv,
                         gate=gate, concat_o=concat_o, q=q, k=k, v=v, ps=ps))
        x = x_out

    x_final = x
    logits = gpt.rms(x_final) @ gpt.head
    loss = ce_loss(logits, targets)

    grads = {'E': np.zeros_like(gpt.E), 'head': np.zeros_like(gpt.head),
             'layers': [dict.fromkeys(W) for W in gpt.layers]}
    p = softmax(logits)
    onehot = np.zeros_like(p)
    onehot[np.arange(T), targets] = 1.0
    dlogits = (p - onehot) / T                              # ← 第 12 课 + 取平均
    grads['head'] = gpt.rms(x_final).T @ dlogits            # ← 模板一
    dx = rms_backward(dlogits @ gpt.head.T, x_final)        # ← 模板二

    for i in reversed(range(gpt.NL)):                       # 逆流水线
        W, t = gpt.layers[i], tape[i]
        # ---- FFN 支路:gate = silu(u)⊙zv ----
        dg = dx @ W['W2'].T
        du = dg * silu_grad(t['u']) * t['zv']               # ← 乘积法则:带上 zv!
        dzv = dg * (t['u'] / (1 + np.exp(-t['u'])))
        grads['layers'][i]['W2'] = t['gate'].T @ dx
        grads['layers'][i]['W1'] = t['x_rms'].T @ du
        grads['layers'][i]['W3'] = t['x_rms'].T @ dzv
        dx_mid = dx + rms_backward(du @ W['W1'].T + dzv @ W['W3'].T, t['x_mid'])
        # ---- attention 支路 ----
        dconcat = dx_mid @ W['Wo'].T
        grads['layers'][i]['Wo'] = t['concat_o'].T @ dx_mid
        dvs, dqs, dks = [], [], []
        for h in range(gpt.H):
            sl = slice(h*gpt.dh, (h+1)*gpt.dh)
            p_h, v_h, q_h, k_h = t['ps'][h], t['v'][:, sl], t['q'][:, sl], t['k'][:, sl]
            dO = dconcat[:, sl]
            dvs.append(p_h.T @ dO)
            dp = dO @ v_h.T
            ds = p_h * (dp - (dp * p_h).sum(-1, keepdims=True))   # softmax Jacobian
            dqs.append(ds @ k_h / math.sqrt(gpt.dh))
            dks.append(ds.T @ q_h / math.sqrt(gpt.dh))
        dv = np.concatenate(dvs, -1)
        dq = rope_backward(np.concatenate(dqs, -1), pos, gpt.D)
        dk = rope_backward(np.concatenate(dks, -1), pos, gpt.D)
        grads['layers'][i]['Wv'] = t['x_in'].T @ dv
        grads['layers'][i]['Wq'] = t['x_in'].T @ dq
        grads['layers'][i]['Wk'] = t['x_in'].T @ dk
        dx = dx_mid + dq @ W['Wq'].T + dk @ W['Wk'].T + dv @ W['Wv'].T

    np.add.at(grads['E'], np.array(ids), dx)                # embedding:scatter-add
    return loss, grads

## 数值裁判(讲义 §4–§5)

In [ ]:
def num_grad(f, x, h=1e-6):
    """f() 返回标量且读的是 x 本体;逐坐标中心差分。"""
    g = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'])
    for _ in it:
        i = it.multi_index
        old = x[i]
        x[i] = old + h; fp = f()
        x[i] = old - h; fm = f()
        x[i] = old
        g[i] = (fp - fm) / (2 * h)
    return g

rng = np.random.default_rng(0)
gpt = MiniGPT()
ids, targets = [1, 2, 3, 4, 5, 6], [2, 3, 4, 5, 6, 7]

x5 = rng.normal(size=(5, 8)); dy5 = rng.normal(size=(5, 8))
print(f"rms_backward  max|err| = "
      f"{np.abs(rms_backward(dy5, x5) - num_grad(lambda: (dy5 * (x5 / np.sqrt((x5**2).mean(-1, keepdims=True)+1e-5))).sum(), x5)).max():.2e}")

xr = rng.normal(size=(6, 8)); dyr = rng.normal(size=(6, 8)); posr = np.arange(6)
print(f"rope_backward max|err| = "
      f"{np.abs(rope_backward(dyr, posr, 8) - num_grad(lambda: (dyr * gpt.rope(xr, posr)).sum(), xr)).max():.2e}")

u0 = rng.normal(size=(4, 6)) * 3
print(f"silu_grad     max|err| = "
      f"{np.abs(silu_grad(u0) - num_grad(lambda: (u0/(1+np.exp(-u0))).sum(), u0)).max():.2e}")

In [ ]:
# 整机验收:taped 前向 == 原版?然后全 33,792 坐标对账(数值基准 = 20 课原版 forward)
lg0, _ = gpt.forward(ids)
loss, grads = loss_and_grads(gpt, ids, targets)
print(f"loss = {loss:.8f}")

def full_loss():
    return ce_loss(gpt.forward(ids)[0], targets)

t0 = time.time()
ana = {'E': grads['E'], 'head': grads['head']}
for li, Wg in enumerate(grads['layers']):
    for name in Wg:
        ana[f'layers[{li}].{name}'] = Wg[name]
num = {'E': num_grad(full_loss, gpt.E), 'head': num_grad(full_loss, gpt.head)}
for li, W in enumerate(gpt.layers):
    for name in W:
        num[f'layers[{li}].{name}'] = num_grad(full_loss, W[name])
max_err, worst = 0.0, None
for path in ana:
    e = np.abs(ana[path] - num[path]).max()
    if e > max_err:
        max_err, worst = e, path
print(f"整机:全 {sum(a.size for a in ana.values()):,} 坐标,"
      f"max|err| = {max_err:.2e} @ {worst},用时 {time.time()-t0:.1f}s")

items = [('E', grads['E']), ('head', grads['head'])]
for li, L in enumerate(grads['layers']):
    for n, g in L.items():
        items.append((f'L{li}.{n}', g))
items.sort(key=lambda kv: -np.abs(kv[1]).max())
print("梯度绝对值 top5:", [(n, round(float(np.abs(g).max()), 4)) for n, g in items[:5]])
print("梯度绝对值 bottom3:", [(n, round(float(np.abs(g).max()), 4)) for n, g in items[-3:]])

In [ ]:
# clip_grad_norm(讲义 §6)+ 反向耗时比
def _iter_grads(grads):
    for g in [grads['E'], grads['head']]:
        yield g
    for L in grads['layers']:
        for g in L.values():
            yield g

def clip_grad_norm(grads, max_norm):
    """全局范数裁剪:超上限等比缩放。返回(裁前, 裁后)。"""
    total = math.sqrt(sum((g**2).sum() for g in _iter_grads(grads)))
    scale = min(1.0, max_norm / (total + 1e-6))
    for g in _iter_grads(grads):
        g *= scale
    return total, total * scale

big = {'E': grads['E']*100, 'head': grads['head']*100,
       'layers': [{k: v*100 for k, v in L.items()} for L in grads['layers']]}
before, after = clip_grad_norm(big, 1.0)
print(f"clip_grad_norm:{before:.3f} → {after:.3f}(上限 1.0)")

t0 = time.time()
for _ in range(200):
    gpt.forward(ids)
t_fwd = (time.time() - t0) / 200
t0 = time.time()
for _ in range(200):
    loss_and_grads(gpt, ids, targets)
t_fb = (time.time() - t0) / 200
print(f"forward {t_fwd*1e3:.2f} ms | fwd+bwd {t_fb*1e3:.2f} ms | 反向/前向 ≈ {t_fb/t_fwd:.1f}×")

In [ ]:
# 鲁棒性:换两套配置抽查坐标(讲义 §5 检验3)
def robust_check(vocab, d, n_heads, n_layers, seed, n_pts=10):
    m = MiniGPT(vocab=vocab, d=d, n_heads=n_heads, n_layers=n_layers, seed=seed)
    rr = np.random.default_rng(seed)
    ii = list(rr.integers(0, vocab, 9)); tt = list(rr.integers(0, vocab, 9))
    _, g2 = loss_and_grads(m, ii, tt)
    gmap = {id(m.E): g2['E'], id(m.head): g2['head']}
    for L, Lg in zip(m.layers, g2['layers']):
        for k in L:
            gmap[id(L[k])] = Lg[k]
    fl = lambda: ce_loss(m.forward(ii)[0], tt)
    errs = []
    for arr in [m.E, m.head] + [W for L in m.layers for W in L.values()]:
        for _ in range(n_pts):
            idx = tuple(rr.integers(0, s, 1)[0] for s in arr.shape)
            old = arr[idx]
            arr[idx] = old + 1e-6; fp = fl()
            arr[idx] = old - 1e-6; fm = fl()
            arr[idx] = old
            errs.append(abs(gmap[id(arr)][idx] - (fp - fm) / 2e-6))
    return max(errs), len(errs)

for cfg in [dict(vocab=20, d=16, n_heads=2, n_layers=1, seed=7),
            dict(vocab=11, d=24, n_heads=3, n_layers=3, seed=42)]:
    e, n = robust_check(**cfg)
    print(f"vocab={cfg['vocab']} d={cfg['d']} {cfg['n_heads']}头{cfg['n_layers']}层:"
          f"抽查 {n} 坐标 max|err| = {e:.2e}")

In [ ]:
# ---- 作业现场 ----

# TODO(C1): 关讲义默写 rms_backward,过数值裁判(阈值 1e-8)

# TODO(C2): 2×3 玩具矩阵验证 softmax Jacobian 行式
# p = softmax(z); dp = 固定随机; ds_ana = p*(dp-(dp*p).sum(-1,keepdims))
# ds_num = num_grad(lambda: (dp*softmax(z)).sum(), z)

# TODO(C3): 默写 clip_grad_norm,验证不超限时梯度不变(scale=1)

# TODO(D1): d=4096 L=32 T=4K batch=8 的激活显存估算

# TODO(D3): 查证 PyTorch retain_graph=True,对照本课 tape

# TODO(E1): batch 化改造(ids 变 [B,T]),B=1 与本题输出逐位对照
